# COMP34812 Coursework - Cross Encoder

## Install dependency

In [ ]:
from pathlib import Path
root_path = Path().resolve().parent.parent
print(f"root_path: {root_path}")

!pip install -r {root_path}/requirements.txt

## Import dependency

In [ ]:
from dataset_av_final import FinalAVDataset
from multi_head_cross_encoder import MultiHeadCrossEncoder
import os
import pandas as pd
import torch
from torch import nn
from torch.optim import AdamW
from torch.utils.data import DataLoader
from transformers import AutoTokenizer

## Global variable

In [ ]:
HEAD_WEIGHTS = torch.tensor([0.3, 0.7], dtype=torch.float)
# The name of the model that both tokenizer and the embedder use (must use the same)
# Consider MODEL_NAME as a specific language used by both tokenizer and the embedder for communication, i.e. they know which token_id represents which token
MODEL_NAME = f"bert-base-uncased"
ROOT_PATH = os.path.dirname(os.path.dirname(os.getcwd()))
CSV_NAME_TRAIN = f"AV_trial_final.csv"
CSV_PATH_TRAIN = os.path.join(ROOT_PATH, "data", "AV", CSV_NAME_TRAIN)
CSV_NAME_TEST = f"dev_final.csv"
CSV_PATH_TEST = os.path.join(ROOT_PATH, "data", "AV", CSV_NAME_TEST)
CROSS_ENCODER_NAME = f"cross-encoder.pth"
CROSS_ENCODER_PATH = os.path.join(ROOT_PATH, "data", "model", CROSS_ENCODER_NAME)
SEPARATION_TOKEN = f"[SEP]"
# Larger batch for efficiency
BATCH_SIZE = 8
SHUFFLE_DATA = False
# TODO: Update
TRAIN_EPOCH = 1
LEARNING_RATE = 2e-5
# Number of train_epoch to log once
EPOCH_PER_LOG = 10

## Utils

In [ ]:
def log(text=None) -> None:
    if text is None:
        print(f"[INFO]")
    else:
        print(f"[INFO] {text}")

## Prepare data

In [ ]:
log(f"Reading csv file at {CSV_PATH_TRAIN} ...")
    
# Load dataset
# Training
dataset_train = FinalAVDataset(CSV_PATH_TRAIN)
dataloader_train = DataLoader(dataset_train, batch_size=BATCH_SIZE, shuffle=SHUFFLE_DATA)
# Testing
# NOTE: Technically, this is from a validation dataset
dataset_test = FinalAVDataset(CSV_PATH_TEST)
dataloader_test = DataLoader(dataset_test, batch_size=BATCH_SIZE, shuffle=SHUFFLE_DATA)

## Instantiate entity

In [ ]:
device = torch.device(f"cuda" if torch.cuda.is_available() else f"cpu")
log(f"Using device {device} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
cross_encoder = MultiHeadCrossEncoder(
    model_name=MODEL_NAME, 
    head_weights=HEAD_WEIGHTS
).to(device)
optimizer = AdamW(cross_encoder.parameters(), lr=LEARNING_RATE)
# This expects raw numbers (logits) and handles the Sigmoid internally
criterion = nn.BCEWithLogitsLoss()

## Train model

In [ ]:
cross_encoder.train()

for epoch in range(TRAIN_EPOCH):
    for batch_X, batch_author_score, batch_style_similarity_score in dataloader_train:
        batch_author_score = batch_author_score.to(device)
        batch_style_similarity_score = batch_style_similarity_score.to(device)
    
        # Reset optimizer for the current gradient descent
        optimizer.zero_grad()

        """
        {
            "input_ids": tensor, 
            "token_type_ids": tensor, 
            "attention_mask": tensor
        }
        """
        encoding_X = tokenizer(
            batch_X[0], 
            batch_X[1], 
            padding=True, 
            truncation=True, 
            return_tensors=f"pt"
        ).to(device)

        y_predictions = cross_encoder(
            input_ids=encoding_X["input_ids"], 
            attention_mask=encoding_X["attention_mask"]
        )

        # Calculate individual losses
        loss_author = criterion(y_predictions["score_author"].squeeze(), batch_author_score)
        loss_style_similarity = criterion(y_predictions["score_style_similarity"].squeeze(), batch_style_similarity_score)

        # Combine loss
        loss = loss_author + loss_style_similarity

        # Apply gradient descent
        loss.backward()
        optimizer.step()
    
    if epoch % EPOCH_PER_LOG == 0:
        # Log
        log(f"[EPOCH {epoch + 1} / {TRAIN_EPOCH}] Loss: {loss.item()}")

## Save trained model

In [ ]:
# Save model weights
torch.save(cross_encoder.state_dict(), CROSS_ENCODER_PATH)

log(f"Model weights are saved to {CROSS_ENCODER_PATH} ...")

## Evaluate model

In [ ]:
cross_encoder = MultiHeadCrossEncoder(model_name=MODEL_NAME, head_weights=HEAD_WEIGHTS).to(device)
cross_encoder.load_state_dict(torch.load(CROSS_ENCODER_PATH))

cross_encoder.eval()

total_loss_test = 0.0

with torch.no_grad():
    for batch_X, batch_author_score, batch_style_similarity_score in dataloader_test:
        batch_author_score = batch_author_score.to(device)
        batch_style_similarity_score = batch_style_similarity_score.to(device)

        """
        {
            "input_ids": tensor, 
            "token_type_ids": tensor, 
            "attention_mask": tensor
        }
        """
        encoding_X = tokenizer(
            batch_X[0], 
            batch_X[1], 
            padding=True, 
            truncation=True, 
            return_tensors=f"pt"
        ).to(device)

        y_predictions = cross_encoder(
            input_ids=encoding_X["input_ids"], 
            attention_mask=encoding_X["attention_mask"]
        ).detach()

        # Calculate individual losses
        loss_author = criterion(y_predictions["score_author"].squeeze(), batch_author_score)
        loss_style_similarity = criterion(y_predictions["score_style_similarity"].squeeze(), batch_style_similarity_score)

        # Combine loss
        loss_test = loss_author + loss_style_similarity

        total_loss_test += loss_test.item()

# Calculate average testing loss
average_loss_test = total_loss_test / len(dataloader_test)
log(f"Testing loss: {average_loss_test:.4f}")